In [1]:



from __future__ import annotations

import csv
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# --- edit ------------------------------------------------------------------
ROOT = "/kaggle/input/datasets/atkiyaanisa/acne04/acne04"
GROUPS_CSV = "fold_audit_out/groups.csv"        # from the fold audit
DUPLICATES_CSV = "fold_audit_out/duplicate_groups.csv"   # from verify_duplicates
OUT = "train_out"

EXCLUDE_DUPLICATES = True
REQUIRE_GROUPS     = True   # refuse to train without a subject-disjoint split
BACKBONE = "resnet18"      # resnet18 | resnet50 | efficientnet_b0
HEAD = "softmax"           # softmax | coral
# The CORAL head squeezed grade 2 into a band of width 1.31 against 2.91 for
# grade 1, giving it a 70.5% error rate that the softmax head cuts to 48.9%.
# The first margin test was therefore measured against errors the architecture
# created, so it has to be repeated on this head before the verdict stands.
IMG_SIZE = 224
EPOCHS = 20
BATCH = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
N_FOLDS = 5
VAL_FRAC = 0.15            # of the training groups, held out for checkpointing
SEED = 0
# ---------------------------------------------------------------------------

CUTS = (5, 20, 50)
N_GRADES = 4
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
GRADE_RE = re.compile(r"levle(\d+)_", re.I)


def rule(count, cuts=CUTS):
    g = 0
    for c in cuts:
        if count > c:
            g += 1
        else:
            break
    return g


def margin_of(count, cuts=CUTS):
    """Lesions away from flipping grade. 0 means adjacent to a cut-point."""
    best = None
    for c in cuts:
        d = abs(count - c) if count <= c else abs(count - (c + 1))
        best = d if best is None else min(best, d)
    return best


# ---------------------------------------------------------------------------
# data
# ---------------------------------------------------------------------------

def load_records(root: Path):
    """{filename: {"grade", "count", "path"}} from the classification files."""
    imgs = {}
    for p in root.rglob("*"):
        if p.suffix.lower() in IMAGE_EXTS and p.is_file():
            if p.name not in imgs or "Classification" in p.parts:
                imgs[p.name] = p

    rec = {}
    for p in root.rglob("*.txt"):
        for line in p.read_text(errors="replace").splitlines():
            parts = line.replace(",", " ").split()
            name = next((t for t in parts if Path(t).suffix.lower() in IMAGE_EXTS), None)
            if not name:
                continue
            ints = []
            for tok in parts:
                if tok != name:
                    try:
                        ints.append(int(float(tok)))
                    except ValueError:
                        pass
            # classification lines carry 1-3 integers (grade, count).
            # Detection box lines carry 4+ and MUST be excluded: a box whose
            # first coordinate happens to fall in 0..3 would pass the grade
            # check below and silently overwrite the label with coordinates.
            if 2 <= len(ints) <= 3 and Path(name).name in imgs:
                g, c = ints[0], ints[1]
                if 0 <= g < N_GRADES and c >= 0:
                    rec[Path(name).name] = {"grade": g, "count": c,
                                            "path": imgs[Path(name).name]}

    if not rec:
        raise SystemExit("no labels parsed - check ROOT and the annotation layout")

    counts = [r["count"] for r in rec.values()]
    print(f"  lesion counts               : {min(counts)}-{max(counts)}, "
          f"mean {sum(counts)/len(counts):.2f}")

    # Self-check using the audit's own result: grade == rule(count) held for
    # 1457/1457 images. If the parse is reading the wrong columns, this breaks
    # immediately instead of training a model on scrambled labels.
    agree = sum(1 for r in rec.values() if rule(r["count"]) == r["grade"])
    frac = agree / len(rec)
    print(f"  grade == rule(count)        : {agree}/{len(rec)} ({frac:.2%})")
    if frac < 0.98:
        raise SystemExit(
            "grade and rule(count) disagree on %.1f%% of images. The week-1 audit "
            "found exact agreement, so the parser is almost certainly reading the "
            "wrong columns. Fix it before training." % (100 * (1 - frac)))
    return rec


USE_IMAGE_CACHE = True
CACHE_DIR = "/kaggle/working/img_cache"
CACHE_SIZE = 256          # shorter side; the model only ever sees 224


def build_image_cache(rec, size=CACHE_SIZE, cache_dir=CACHE_DIR):
    """
    Resize every image once, then train from the cache.

    ACNE04 images are 3112x3456. Decoding ~10-megapixel JPEGs every epoch makes
    training DATA-bound rather than GPU-bound - earlier runs spent two hours
    with the GPU largely idle, waiting on libjpeg. Caching at 256px cuts that
    to minutes. The model only ever sees 224x224 crops, so nothing it can
    observe changes.

    draft() lets libjpeg decode at a reduced scale directly, which is far
    faster than decoding full size and then shrinking.
    """
    from PIL import Image
    d = Path(cache_dir)
    d.mkdir(parents=True, exist_ok=True)
    made = reused = failed = 0
    for n, r in rec.items():
        dst = d / (Path(n).stem + ".jpg")
        if dst.exists():
            reused += 1
        else:
            try:
                with Image.open(r["path"]) as im:
                    im.draft("RGB", (size, size))
                    im = im.convert("RGB")
                    w, h = im.size
                    s = size / min(w, h)
                    if s < 1.0:
                        im = im.resize((max(1, round(w * s)), max(1, round(h * s))),
                                       Image.BILINEAR)
                    im.save(dst, "JPEG", quality=92)
                made += 1
            except Exception as e:
                failed += 1
                print(f"    cache failed for {n}: {e}")
                continue
        r["path"] = dst
    print(f"  image cache                 : {made} built, {reused} reused"
          + (f", {failed} FAILED (using originals)" if failed else "")
          + f"  -> {d}")
    return rec


def find_file(name, hint=None):
    """
    Locate an audit CSV wherever Kaggle actually put it.

    The audits write to ./fold_audit_out/ inside THEIR OWN session. Attaching
    that notebook to a new one mounts it read-only under
    /kaggle/input/<slug>/fold_audit_out/ instead, so the literal relative path
    does not exist. Searching for it removes the single most common failure.
    """
    cands = []
    if hint:
        cands.append(Path(hint))
    cands += [Path(name), Path("fold_audit_out") / name,
              Path("/kaggle/working/fold_audit_out") / name]
    for c in cands:
        if c.is_file():
            return c
    for base in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if base.is_dir():
            try:
                hits = sorted(base.rglob(name))
            except OSError:
                hits = []
            if hits:
                print(f"  found {name} at {hits[0]}")
                return hits[0]
    return None


def load_groups(path, names):
    """Subject group per image; singleton groups for anything missing."""
    g = {}
    try:
        with open(path) as fh:
            for r in csv.DictReader(fh):
                g[r["image"]] = r["subject_group"]
    except (FileNotFoundError, TypeError):
        print("\n" + "!" * 70)
        print(f"  groups.csv NOT FOUND ({path}).")
        print("  Every image would become its own subject, so near-duplicates of the")
        print("  same face could land in train AND test. That is the exact leakage")
        print("  this project exists to measure - reporting numbers from such a split")
        print("  would repeat the mistake the audit documents.")
        print("  Attach the fold-audit notebook's output, or set REQUIRE_GROUPS=False")
        print("  to proceed anyway with the split clearly labelled as ungrouped.")
        print("!" * 70)
        if REQUIRE_GROUPS:
            raise SystemExit(
                "stopping: refusing to train on a split that is not subject-disjoint")
    for i, n in enumerate(names):
        g.setdefault(n, f"_solo_{i}")
    return g


def load_duplicate_drops(path):
    """Keep the first filename in each duplicate group, drop the rest."""
    drop = set()
    try:
        with open(path) as fh:
            by_group = defaultdict(list)
            for r in csv.DictReader(fh):
                by_group[r["group"]].append(r["image"])
        for _, members in by_group.items():
            for m in sorted(members)[1:]:
                drop.add(m)
    except (FileNotFoundError, TypeError):
        print("  NOTE: duplicate_groups.csv not found - no duplicates removed.")
        print("  Run verify_duplicates first if you want the 38 groups excluded.")
    return drop


def group_kfold(names, group_of, k, seed=SEED):
    """Assign whole subject groups to folds, largest groups first for balance."""
    by_group = defaultdict(list)
    for n in names:
        by_group[group_of[n]].append(n)
    groups = sorted(by_group, key=lambda g: (-len(by_group[g]), str(g)))
    rng = random.Random(seed)
    rng.shuffle(groups)
    groups.sort(key=lambda g: -len(by_group[g]))

    sizes = [0] * k
    assign = {}
    for g in groups:
        f = min(range(k), key=lambda i: sizes[i])
        assign[g] = f
        sizes[f] += len(by_group[g])
    return {n: assign[group_of[n]] for n in names}


# ---------------------------------------------------------------------------
# metrics
# ---------------------------------------------------------------------------

def qwk(y_true, y_pred, k=N_GRADES):
    O = np.zeros((k, k))
    for t, p in zip(y_true, y_pred):
        O[t, p] += 1
    w = np.array([[((i - j) ** 2) / ((k - 1) ** 2) for j in range(k)] for i in range(k)])
    ht = np.bincount(y_true, minlength=k)
    hp = np.bincount(y_pred, minlength=k)
    E = np.outer(ht, hp) / max(1, len(y_true))
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0


def macro_f1(y_true, y_pred, k=N_GRADES):
    fs = []
    for c in range(k):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        fs.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(fs))


def risk_coverage(conf, loss):
    """
    AURC and AUGRC.

    Sort by confidence descending, accept the top k, and track two risks:
      selective risk  = mean loss over the accepted (the classic, flawed one)
      generalized risk = summed loss over accepted, divided by N

    Traub et al. (NeurIPS 2024) argue AURC misranks methods and propose AUGRC.
    Report both; make AUGRC primary.
    """
    conf = np.asarray(conf, dtype=float)
    loss = np.asarray(loss, dtype=float)
    n = len(conf)
    order = np.argsort(-conf)
    cum = np.cumsum(loss[order])
    ks = np.arange(1, n + 1)
    sel = cum / ks
    gen = cum / n
    return float(sel.mean()), float(gen.mean()), (ks / n, sel, gen)


def auroc(scores, labels):
    """Rank-based AUROC, ties at 0.5. Higher score should mean label 1."""
    s = np.asarray(scores, dtype=float)
    y = np.asarray(labels, dtype=int)
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    allv = np.concatenate([pos, neg])
    order = np.argsort(allv, kind="mergesort")
    ranks = np.empty(len(allv), dtype=float)
    ranks[order] = np.arange(1, len(allv) + 1)
    i = 0
    sv = allv[order]
    while i < len(sv):
        j = i
        while j + 1 < len(sv) and sv[j + 1] == sv[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = (i + 1 + j + 1) / 2
        i = j + 1
    return float((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2)
                 / (len(pos) * len(neg)))


def bootstrap_ci(fn, *arrays, groups=None, n_boot=1000, seed=SEED):
    """
    Cluster bootstrap over subject groups, because images of one subject are
    not independent. Returns (lo, hi) at 95%.
    """
    rng = np.random.default_rng(seed)
    arrays = [np.asarray(a) for a in arrays]
    n = len(arrays[0])
    if groups is None:
        groups = np.arange(n)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    idx_by_group = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_group[g] for g in pick])
        try:
            v = fn(*[a[idx] for a in arrays])
            if not (isinstance(v, float) and math.isnan(v)):
                vals.append(v)
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


# ---------------------------------------------------------------------------
# model
# ---------------------------------------------------------------------------

ALLOW_RANDOM_INIT = False   # see the message below before setting this True


def _weights(fn, w):
    """
    Fetch pretrained weights, and explain clearly if the network is off.

    Kaggle notebooks default Internet to OFF, so the torchvision download
    fails with a bare DNS error that says nothing about the real cause.
    """
    import urllib.error, socket
    try:
        return fn(weights=w)
    except (urllib.error.URLError, socket.gaierror, OSError) as e:
        print("\n" + "!" * 70)
        print("  Could not download the pretrained backbone weights.")
        print(f"  ({type(e).__name__}: {e})")
        print()
        print("  Kaggle notebooks start with Internet turned OFF.")
        print("  Fix: right sidebar -> Settings -> Internet -> On, then Run All.")
        print()
        print("  Training from random initialisation instead would change every")
        print("  number and make this run non-comparable with the others, so it")
        print("  is refused by default. Set ALLOW_RANDOM_INIT = True only if you")
        print("  accept that, and label those results as not pretrained.")
        print("!" * 70)
        if not ALLOW_RANDOM_INIT:
            raise SystemExit("stopping: enable Internet in notebook settings")
        print("  ALLOW_RANDOM_INIT is True - continuing WITHOUT pretrained weights.")
        return fn(weights=None)


def build_model(backbone, device, priors=None):
    import torch
    import torchvision

    if backbone == "resnet18":
        m = _weights(torchvision.models.resnet18, torchvision.models.ResNet18_Weights.DEFAULT)
        feat = m.fc.in_features
        m.fc = torch.nn.Identity()
    elif backbone == "resnet50":
        m = _weights(torchvision.models.resnet50, torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        feat = m.fc.in_features
        m.fc = torch.nn.Identity()
    else:
        m = _weights(torchvision.models.efficientnet_b0,
            torchvision.models.EfficientNet_B0_Weights.DEFAULT)
        feat = m.classifier[1].in_features
        m.classifier = torch.nn.Identity()

    class Coral(torch.nn.Module):
        """
        CORAL ordinal head: one shared projection plus K-1 independent biases,
        which guarantees the predicted cumulative probabilities are monotone.
        """
        def __init__(self, backbone, feat, k, priors=None):
            super().__init__()
            self.backbone = backbone
            self.fc = torch.nn.Linear(feat, 1, bias=False)
            # Zero-initialised biases are the trap: all K-1 thresholds start on
            # top of each other, the middle grades get almost no probability
            # mass, and the head can stay collapsed for the whole run. Start
            # them at the label marginals instead, so sigmoid(bias_k) equals
            # the training P(y > k) before a single image is seen.
            if priors is None:
                b = torch.zeros(k - 1)
            else:
                q = torch.tensor(priors, dtype=torch.float32).clamp(1e-3, 1 - 1e-3)
                b = torch.log(q / (1 - q))
            self.bias = torch.nn.Parameter(b)

        def forward(self, x):
            return self.fc(self.backbone(x)) + self.bias

    class Softmax(torch.nn.Module):
        """One logit per grade; no shared scalar, so no class can be squeezed."""
        def __init__(self, backbone, feat, k, priors=None):
            super().__init__()
            self.backbone = backbone
            self.fc = torch.nn.Linear(feat, k)
            if priors is not None:
                q = [1.0 - priors[0]]
                for i in range(len(priors) - 1):
                    q.append(priors[i] - priors[i + 1])
                q.append(priors[-1])
                q = torch.tensor(q, dtype=torch.float32).clamp(1e-3, 1.0)
                with torch.no_grad():
                    self.fc.bias.copy_(torch.log(q / q.sum()))

        def forward(self, x):
            return self.fc(self.backbone(x))

    if HEAD == "softmax":
        return Softmax(m, feat, N_GRADES, priors)
    return Coral(m, feat, N_GRADES, priors)


def coral_targets(y, k=N_GRADES):
    import torch
    return (y.unsqueeze(1) > torch.arange(k - 1, device=y.device).unsqueeze(0)).float()


def coral_to_dist(logits):
    """
    Cumulative sigmoids -> a proper distribution over grades.

    P(y>0), P(y>1), P(y>2) are enforced monotone by taking a running minimum,
    then differenced. Without that, a non-monotone triple yields negative
    probabilities and every downstream confidence score is meaningless.
    """
    p = 1.0 / (1.0 + np.exp(-logits))
    p = np.minimum.accumulate(p, axis=1)
    cum = np.concatenate([np.ones((len(p), 1)), p, np.zeros((len(p), 1))], axis=1)
    d = cum[:, :-1] - cum[:, 1:]
    d = np.clip(d, 1e-9, None)
    return d / d.sum(axis=1, keepdims=True)


def softmax_dist(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def to_dist(logits):
    return softmax_dist(logits) if HEAD == "softmax" else coral_to_dist(logits)


def predict_from_logits(logits):
    """
    One decision rule, used for BOTH checkpoint selection and the test set.

    This is CORAL's threshold rule: count how many cumulative probabilities
    exceed 0.5. Do NOT use argmax of the differenced distribution - its middle
    entries are DIFFERENCES between adjacent cumulative probabilities, so they
    are structurally small, and argmax collapses onto the two end classes
    whenever the biases are not widely separated.
    """
    if HEAD == "softmax":
        return logits.argmax(axis=1).astype(int)
    p = 1.0 / (1.0 + np.exp(-logits))
    p = np.minimum.accumulate(p, axis=1)
    return (p > 0.5).sum(axis=1).astype(int)


# ---------------------------------------------------------------------------
# training
# ---------------------------------------------------------------------------

def run_fold(rec, names, fold_of, groups_of, fold, device, log):
    import torch
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as T
    from PIL import Image

    test = [n for n in names if fold_of[n] == fold]
    pool = [n for n in names if fold_of[n] != fold]

    # carve a validation set out of the training pool, by whole subject groups
    pool_groups = sorted({groups_of[n] for n in pool})
    rng = random.Random(SEED + fold)
    rng.shuffle(pool_groups)
    n_val = max(1, int(len(pool_groups) * VAL_FRAC))
    val_groups = set(pool_groups[:n_val])
    val = [n for n in pool if groups_of[n] in val_groups]
    train = [n for n in pool if groups_of[n] not in val_groups]

    assert not ({groups_of[n] for n in train} & {groups_of[n] for n in val})
    assert not ({groups_of[n] for n in train} & {groups_of[n] for n in test})
    print(f"\n  fold {fold}: train {len(train)}  val {len(val)}  test {len(test)}")

    tf_train = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.2, 0.2, 0.2, 0.05),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    tf_eval = T.Compose([
        T.Resize(int(IMG_SIZE * 1.14)), T.CenterCrop(IMG_SIZE), T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    class DS(Dataset):
        def __init__(self, items, tf):
            self.items, self.tf = items, tf

        def __len__(self):
            return len(self.items)

        def __getitem__(self, i):
            n = self.items[i]
            img = Image.open(rec[n]["path"]).convert("RGB")
            return self.tf(img), rec[n]["grade"], i

    dl_train = DataLoader(DS(train, tf_train), batch_size=BATCH, shuffle=True,
                          num_workers=2, drop_last=False)
    dl_val = DataLoader(DS(val, tf_eval), batch_size=BATCH, num_workers=2)
    dl_test = DataLoader(DS(test, tf_eval), batch_size=BATCH, num_workers=2)

    tr_y = [rec[n]["grade"] for n in train]
    priors = [sum(1 for v in tr_y if v > kk) / max(1, len(tr_y))
              for kk in range(N_GRADES - 1)]
    print(f"    P(y>k) in train: {[round(p,3) for p in priors]}")

    model = build_model(BACKBONE, device, priors).to(device)
    # Weight decay must NOT touch the ordinal thresholds: shrinking them toward
    # zero is exactly the collapse this head has to avoid.
    if HEAD == "softmax":
        nod = [p for nm, p in model.named_parameters() if nm.endswith("bias")]
        dec = [p for nm, p in model.named_parameters() if not nm.endswith("bias")]
        opt = torch.optim.AdamW(
            [{"params": dec, "weight_decay": WEIGHT_DECAY},
             {"params": nod, "weight_decay": 0.0}], lr=LR)
    else:
        decay = [p for n_, p in model.named_parameters() if n_ != "bias"]
        opt = torch.optim.AdamW(
            [{"params": decay, "weight_decay": WEIGHT_DECAY},
             {"params": [model.bias], "weight_decay": 0.0, "lr": LR * 10}],
            lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    lossf = torch.nn.BCEWithLogitsLoss()
    celoss = torch.nn.CrossEntropyLoss()

    def evaluate(dl):
        model.eval()
        L, Y = [], []
        with torch.no_grad():
            for x, y, _ in dl:
                L.append(model(x.to(device)).cpu().numpy())
                Y.append(y.numpy())
        return np.concatenate(L), np.concatenate(Y)

    best_qwk, best_state, best_ep = -2.0, None, -1
    for ep in range(EPOCHS):
        model.train()
        tot = 0.0
        for x, y, _ in dl_train:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = (celoss(out, y) if HEAD == "softmax"
                    else lossf(out, coral_targets(y)))
            loss.backward()
            opt.step()
            tot += loss.item() * len(y)
        sched.step()

        vl, vy = evaluate(dl_val)
        v = qwk(vy, predict_from_logits(vl))
        flag = ""
        if v > best_qwk:                       # VALIDATION qwk, not train accuracy
            best_qwk, best_ep = v, ep
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
            flag = "  <- best"
        print(f"    ep {ep:>2}  train loss {tot/max(1,len(train)):.4f}  val qwk {v:.4f}{flag}")

    print(f"    restoring epoch {best_ep} (val qwk {best_qwk:.4f})")
    model.load_state_dict(best_state)
    if HEAD != "softmax":
        print(f"    learned thresholds (bias): "
              f"{[round(float(v),3) for v in model.bias.detach().cpu()]}")

    tl, ty = evaluate(dl_test)
    dist = to_dist(tl)
    pred = predict_from_logits(tl)

    seen = sorted(set(pred.tolist()))
    print(f"    predicted grades: {dict(sorted(Counter(pred.tolist()).items()))}")
    if len(seen) < len(set(ty.tolist())):
        print(f"    *** WARNING: model predicts only {seen} but the test set "
              f"contains {sorted(set(ty.tolist()))}.")
        print("    *** The ordinal head has collapsed. Do not interpret the "
              "error-detection results from this run.")

    for i, n in enumerate(test):
        d = dist[i]
        log.append({
            "image": n, "fold": fold, "subject_group": groups_of[n],
            "true_grade": int(rec[n]["grade"]), "true_count": int(rec[n]["count"]),
            "pred_grade": int(pred[i]),
            "p0": d[0], "p1": d[1], "p2": d[2], "p3": d[3],
            "max_softmax": float(d.max()),
            "p_pred": float(d[int(pred[i])]),
            "entropy": float(-(d * np.log(d)).sum()),
            "coral_margin": float(
                np.partition(d, -2)[-1] - np.partition(d, -2)[-2]
                if HEAD == "softmax" else np.abs(tl[i]).min()),
            "true_margin": int(margin_of(rec[n]["count"])),
            "abs_error": int(abs(int(pred[i]) - int(rec[n]["grade"]))),
            "val_qwk": best_qwk, "best_epoch": best_ep,
        })
    return qwk(ty, pred), float((ty == pred).mean())


# ---------------------------------------------------------------------------
# the decisive analysis
# ---------------------------------------------------------------------------

def analyse(log, out: Path):
    """
    Does the oracle boundary margin detect grading errors better than plain
    confidence? This is the gate the whole method rests on.
    """
    L = {k: np.array([r[k] for r in log]) for k in log[0]}
    n = len(log)
    err01 = (L["abs_error"] > 0).astype(int)
    ordinal_loss = L["abs_error"].astype(float)
    groups = L["subject_group"]

    print("\n" + "=" * 74)
    print("GRADER PERFORMANCE  (pooled out-of-fold, every image once)")
    print("=" * 74)
    k = qwk(L["true_grade"], L["pred_grade"])
    acc = float((L["true_grade"] == L["pred_grade"]).mean())
    f1 = macro_f1(L["true_grade"], L["pred_grade"])
    mae = float(ordinal_loss.mean())
    klo, khi = bootstrap_ci(qwk, L["true_grade"], L["pred_grade"], groups=groups)
    print(f"  n={n}  QWK {k:.4f} [{klo:.4f}, {khi:.4f}]  acc {acc:.4f}  "
          f"macro-F1 {f1:.4f}  MAE {mae:.4f}")
    print(f"  error rate {err01.mean():.1%}   severe errors (|d|>=2) "
          f"{(L['abs_error'] >= 2).mean():.1%}  ({int((L['abs_error']>=2).sum())} images)")

    # confidence scores: higher = more confident
    scores = {
        "max_softmax": L["max_softmax"],
        "neg_entropy": -L["entropy"],
        ("top2_gap" if HEAD == "softmax" else "coral_margin"): L["coral_margin"],
        "true_margin": L["true_margin"].astype(float),
        "margin+softmax": None,       # filled below
    }

    z = lambda v: (v - np.mean(v)) / (np.std(v) + 1e-9)

    def crossfit_combo(feats):
        """
        Learn the combination instead of fixing it, and learn it OUT OF FOLD.

        A fixed equal-weight sum of standardised margin and confidence loses to
        confidence alone whenever confidence is the stronger term - which says
        nothing about whether margin carries information. Fitting the weights
        on the same rows being scored is the opposite error. So: fit the
        combiner on the other folds, score this fold, repeat.
        """
        try:
            from sklearn.linear_model import LogisticRegression
        except Exception:
            return None
        X = np.column_stack([z(f) for f in feats])
        s = np.zeros(len(X))
        for f in np.unique(L["fold"]):
            te = L["fold"] == f
            tr = ~te
            if err01[tr].min() == err01[tr].max():
                s[te] = 0.0
                continue
            m = LogisticRegression(max_iter=2000).fit(X[tr], err01[tr])
            s[te] = -m.predict_proba(X[te])[:, 1]      # high = confident
        return s

    combo = crossfit_combo([L["true_margin"].astype(float), L["max_softmax"]])
    if combo is None:
        scores.pop("margin+softmax")
    else:
        scores["margin+softmax"] = combo

    print("\n" + "=" * 74)
    print("ERROR DETECTION  (AUGRC primary; lower is better for AURC/AUGRC)")
    print("=" * 74)
    print(f"  {'score':>16} {'AUROC':>8} {'AUROC 95% CI':>20} "
          f"{'AURC':>8} {'AUGRC':>8}")
    results = {}
    for name, s in scores.items():
        a = auroc(-s, err01)                 # low confidence should flag errors
        lo, hi = bootstrap_ci(lambda x, y: auroc(-x, y), s, err01, groups=groups)
        aurc, augrc, _ = risk_coverage(s, ordinal_loss)
        results[name] = {"auroc": a, "auroc_lo": lo, "auroc_hi": hi,
                         "aurc": aurc, "augrc": augrc}
        print(f"  {name:>16} {a:>8.4f} [{lo:>7.4f}, {hi:>7.4f}] "
              f"{aurc:>8.4f} {augrc:>8.4f}")

    print("\n" + "-" * 74)
    print("RISK AT FIXED COVERAGE  (ordinal risk = mean |grade error| on accepted)")
    print("-" * 74)
    print(f"  {'score':>16}" + "".join(f"{int(c*100):>10}%" for c in (0.9, 0.8, 0.7, 0.5)))
    for name, s in scores.items():
        row = []
        order = np.argsort(-np.asarray(s, dtype=float))
        for c in (0.9, 0.8, 0.7, 0.5):
            kk = max(1, int(n * c))
            row.append(ordinal_loss[order[:kk]].mean())
        results[name]["risk_at_coverage"] = {str(c): v for c, v in
                                             zip((0.9, 0.8, 0.7, 0.5), row)}
        print(f"  {name:>16}" + "".join(f"{v:>11.4f}" for v in row))

    # ---- does margin add anything beyond confidence? ---------------------
    print("\n" + "=" * 74)
    print("DOES MARGIN ADD INFORMATION BEYOND CONFIDENCE?")
    print("=" * 74)
    lrt = None
    try:
        from sklearn.linear_model import LogisticRegression
        from scipy import stats

        X1 = z(L["max_softmax"]).reshape(-1, 1)
        X2 = np.column_stack([z(L["max_softmax"]), z(L["true_margin"].astype(float))])

        def ll(X):
            m = LogisticRegression(max_iter=2000).fit(X, err01)
            p = np.clip(m.predict_proba(X)[:, 1], 1e-9, 1 - 1e-9)
            return float((err01 * np.log(p) + (1 - err01) * np.log(1 - p)).sum())

        l1, l2 = ll(X1), ll(X2)
        stat = 2 * (l2 - l1)
        p = float(1 - stats.chi2.cdf(stat, df=1))
        lrt = {"ll_softmax": l1, "ll_softmax_plus_margin": l2,
               "lr_stat": stat, "df": 1, "p_value": p}
        print(f"  log-likelihood, softmax only          : {l1:.3f}")
        print(f"  log-likelihood, softmax + margin      : {l2:.3f}")
        print(f"  likelihood-ratio statistic (df=1)     : {stat:.3f}")
        print(f"  p-value                               : {p:.5f}")
        print("\n  " + ("Margin explains error variance that confidence does not."
                        if p < 0.05 else
                        "Margin adds nothing detectable beyond confidence."))
    except Exception as e:
        print(f"  (skipped: {e})")

    # ---- stratify, because abstention must not dump the rare grades ------
    print("\n" + "-" * 74)
    print("BY TRUE GRADE  (a signal that only abstains on rare grades is unfair)")
    print("-" * 74)
    print(f"  {'grade':>5} {'n':>5} {'error rate':>11} {'median margin':>14}")
    by_grade = {}
    for g in range(N_GRADES):
        m = L["true_grade"] == g
        if not m.any():
            continue
        by_grade[g] = {"n": int(m.sum()), "error_rate": float(err01[m].mean()),
                       "median_margin": float(np.median(L["true_margin"][m]))}
        print(f"  {g:>5} {int(m.sum()):>5} {err01[m].mean():>10.1%} "
              f"{np.median(L['true_margin'][m]):>14.1f}")

    print("\n" + "-" * 74)
    print("ERROR RATE BY MARGIN  (the mechanism, if there is one)")
    print("-" * 74)
    print(f"  {'margin':>8} {'n':>5} {'error rate':>11}")
    by_margin = {}
    for mv in sorted(set(L["true_margin"].tolist()))[:10]:
        m = L["true_margin"] == mv
        by_margin[int(mv)] = {"n": int(m.sum()), "error_rate": float(err01[m].mean())}
        print(f"  {mv:>8} {int(m.sum()):>5} {err01[m].mean():>10.1%}")

    print("\n" + "=" * 74)
    print("VERDICT")
    print("=" * 74)
    base = results["max_softmax"]["augrc"]
    best = min((v["augrc"], kk) for kk, v in results.items())
    print(f"  max-softmax AUGRC        : {base:.4f}")

    delta = None
    if "margin+softmax" in results:
        comb = results["margin+softmax"]["augrc"]
        print(f"  margin+softmax AUGRC     : {comb:.4f}  (cross-fitted combiner)")

        # paired bootstrap of the DIFFERENCE, which is the right test:
        # comparing two separate confidence intervals is both weaker and
        # systematically misleading when the two scores are correlated.
        def d_augrc(s_base, s_comb, loss):
            _, g1, _ = risk_coverage(s_base, loss)
            _, g2, _ = risk_coverage(s_comb, loss)
            return g1 - g2                      # positive = combination better

        dlo, dhi = bootstrap_ci(d_augrc, scores["max_softmax"],
                                scores["margin+softmax"], ordinal_loss,
                                groups=groups)
        point = d_augrc(scores["max_softmax"], scores["margin+softmax"], ordinal_loss)
        delta = {"point": point, "lo": dlo, "hi": dhi}
        print(f"  AUGRC improvement        : {point:+.4f} "
              f"[{dlo:+.4f}, {dhi:+.4f}]  (positive favours margin)")
    print(f"  best overall             : {best[1]} ({best[0]:.4f})")

    helps = (delta is not None and delta["lo"] > 0) and (lrt is None or lrt["p_value"] < 0.05)
    if helps:
        print("\n  The ORACLE margin helps. A detector-estimated margin will help less,")
        print("  so treat this as the ceiling and build the detector to find out how")
        print("  much of it survives estimation.")
    elif lrt is not None and lrt["p_value"] < 0.05:
        print("\n  Mixed. Margin carries information about errors (the likelihood-ratio")
        print("  test is significant) but does not translate into a better risk-coverage")
        print("  trade-off than confidence alone. That is a reportable finding in itself:")
        print("  label-structure information is real but redundant with confidence here.")
        print("  Before concluding, try a better combiner and check whether margin helps")
        print("  specifically on the boundary-adjacent subset.")
    else:
        print("\n  The ORACLE margin does not help, and it is the best case: a detector's")
        print("  estimate is strictly noisier. The boundary-margin method is dead on")
        print("  this dataset. Report the negative result and let the audit carry the")
        print("  paper. Do not spend weeks building a detector to confirm it.")
    print("=" * 74)

    out.mkdir(parents=True, exist_ok=True)
    (out / "analysis.json").write_text(json.dumps({
        "n": n, "qwk": k, "qwk_ci": [klo, khi], "accuracy": acc, "macro_f1": f1,
        "mae": mae, "error_rate": float(err01.mean()),
        "severe_error_rate": float((L["abs_error"] >= 2).mean()),
        "error_detection": results, "likelihood_ratio_test": lrt,
        "augrc_improvement": delta,
        "by_grade": by_grade, "by_margin": by_margin,
        "config": {"backbone": BACKBONE, "epochs": EPOCHS, "folds": N_FOLDS,
                   "exclude_duplicates": EXCLUDE_DUPLICATES, "seed": SEED},
    }, indent=2, default=float))
    return results


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def main():
    import torch

    root, out = Path(ROOT), Path(OUT)
    out.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print("=" * 74)
    print(f"GRADER ({HEAD} head) - group-disjoint cross-fitting")
    print("=" * 74)

    rec = load_records(root)
    print(f"  images with grade and count : {len(rec)}")

    if EXCLUDE_DUPLICATES:
        drop = load_duplicate_drops(find_file("duplicate_groups.csv", DUPLICATES_CSV))
        before = len(rec)
        rec = {k: v for k, v in rec.items() if k not in drop}
        print(f"  duplicates dropped          : {before - len(rec)}")
    names = sorted(rec)
    print(f"  distinct images used        : {len(names)}")

    if USE_IMAGE_CACHE:
        rec = build_image_cache(rec)

    groups_of = load_groups(find_file("groups.csv", GROUPS_CSV), names)
    n_groups = len({groups_of[n] for n in names})
    print(f"  subject groups              : {n_groups}")
    print(f"  grade distribution          : "
          f"{dict(sorted(Counter(rec[n]['grade'] for n in names).items()))}")

    fold_of = group_kfold(names, groups_of, N_FOLDS)
    print(f"  fold sizes                  : "
          f"{dict(sorted(Counter(fold_of.values()).items()))}")

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  device                      : {dev}")
    if dev == "cpu":
        print("  WARNING: no GPU. This will take hours. Turn on the accelerator.")

    log, fold_stats = [], []
    for f in range(N_FOLDS):
        k, a = run_fold(rec, names, fold_of, groups_of, f, dev, log)
        fold_stats.append({"fold": f, "qwk": k, "acc": a})
        print(f"    fold {f} test QWK {k:.4f}  acc {a:.4f}")

    cols = list(log[0])
    with (out / "per_image_log.csv").open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols)
        w.writeheader()
        w.writerows(log)
    print(f"\n  wrote {out/'per_image_log.csv'}  ({len(log)} rows)")

    print("\n  per-fold test QWK: " +
          ", ".join(f"{s['qwk']:.4f}" for s in fold_stats))

    analyse(log, out)
    print(f"  wrote {out/'analysis.json'}")
    return 0


main()


GRADER (softmax head) - group-disjoint cross-fitting
  lesion counts               : 1-65, mean 12.95
  grade == rule(count)        : 1457/1457 (100.00%)
  images with grade and count : 1457
  found duplicate_groups.csv at /kaggle/input/notebooks/atkiyaanisa/verifyduplicates-ipynb/fold_audit_out/duplicate_groups.csv
  duplicates dropped          : 38
  distinct images used        : 1419
  image cache                 : 1419 built, 0 reused  -> /kaggle/working/img_cache
  found groups.csv at /kaggle/input/notebooks/atkiyaanisa/foldaudit-ipynb/fold_audit_out/groups.csv
  subject groups              : 1397
  grade distribution          : {0: 503, 1: 612, 2: 176, 3: 128}
  fold sizes                  : {0: 284, 1: 284, 2: 284, 3: 284, 4: 283}
  device                      : cuda

  fold 0: train 967  val 168  test 284
    P(y>k) in train: [0.641, 0.216, 0.09]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


    ep  0  train loss 0.9414  val qwk 0.7273  <- best
    ep  1  train loss 0.6626  val qwk 0.7921  <- best
    ep  2  train loss 0.5801  val qwk 0.7769
    ep  3  train loss 0.5106  val qwk 0.7383
    ep  4  train loss 0.4190  val qwk 0.7167
    ep  5  train loss 0.3742  val qwk 0.8394  <- best
    ep  6  train loss 0.3426  val qwk 0.8467  <- best
    ep  7  train loss 0.2478  val qwk 0.7019
    ep  8  train loss 0.2719  val qwk 0.8606  <- best
    ep  9  train loss 0.2074  val qwk 0.8606  <- best
    ep 10  train loss 0.1773  val qwk 0.8561
    ep 11  train loss 0.1383  val qwk 0.8340
    ep 12  train loss 0.1010  val qwk 0.8249
    ep 13  train loss 0.1088  val qwk 0.8421
    ep 14  train loss 0.0896  val qwk 0.8487
    ep 15  train loss 0.0631  val qwk 0.8315
    ep 16  train loss 0.0515  val qwk 0.8296
    ep 17  train loss 0.0616  val qwk 0.8258
    ep 18  train loss 0.0582  val qwk 0.8433
    ep 19  train loss 0.0441  val qwk 0.8289
    restoring epoch 9 (val qwk 0.8606)
    pre

0